# Lab 09 — Relational Queries — Chinook Music DB
**SQL for Analysts Track** · Beginner–Intermediate · ~50 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Connect to SQLite and list schema tables
2. Write JOIN + GROUP BY revenue queries
3. Use LEFT JOIN anti-patterns to find orphan rows
4. Load SQL results into pandas and plot monthly revenue

## Datasets (this folder)
- `chinook.sqlite` — auto-download from `https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-01-foundations-python-math-data/labs/lab-09-chinook-sqlite/lab-09-chinook-sqlite.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`chinook.sqlite`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-09-chinook-sqlite"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-01-foundations-python-math-data/bundles/lab-09-chinook-sqlite/dataset.zip"
NEED = ["chinook.sqlite"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## SQL for Analysts Track: JOIN, GROUP BY, Results → pandas

> **Scenario:** `chinook.sqlite` is a music-store schema (artists, albums, tracks, invoices, customers). Answer revenue and customer questions with SQL, then load results into pandas for a monthly trend.
>
> **You will learn:** `sqlite3.connect`, SELECT / JOIN / GROUP BY, subqueries, `pandas.read_sql`.
> **Time:** ~50 minutes. **Level:** Beginner–Intermediate. **Needs:** sqlite3 (stdlib) + pandas. **Env:** 🟢 Colab only.

### SQL mental map

| Spreadsheet idea | SQL | Python |
|---|---|---|
| Filter rows | `WHERE` | `df[df.col == …]` |
| VLOOKUP across sheets | `JOIN` | `merge` |
| PivotTable values | `SUM(...) GROUP BY` | `groupby().sum()` |
| Top N | `ORDER BY … LIMIT n` | `nlargest` |
| Two-step pivot | subquery / CTE | chain groupbys |

---

### 1. Connect and list tables (local first, Colab fallback)

In [ ]:
import os, sqlite3
import pandas as pd

def connect_chinook():
    local = "chinook.sqlite"
    if not os.path.exists(local):
        import urllib.request
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/lerocha/chinook-database/master/"
            "ChinookDatabase/DataSources/Chinook_Sqlite.sqlite",
            local,
        )
    return sqlite3.connect(local)

conn = connect_chinook()
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)["name"].tolist()
print(tables)
# ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice',
#  'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']

print(pd.read_sql("SELECT COUNT(*) AS n_customers FROM Customer", conn))
# 59
print(pd.read_sql("SELECT ROUND(SUM(Total),2) AS revenue FROM Invoice", conn))
# 2328.60


Schema chain for track revenue:

```text
Invoice ──< InvoiceLine >── Track ──< Album >── Artist
```

---

### 2. Top 5 tracks by revenue

In [ ]:
top5 = pd.read_sql("""
SELECT t.Name        AS track,
       al.Title      AS album,
       ar.Name       AS artist,
       ROUND(SUM(il.Quantity * il.UnitPrice), 2) AS revenue
FROM InvoiceLine il
JOIN Track t    ON t.TrackId  = il.TrackId
JOIN Album al   ON al.AlbumId = t.AlbumId
JOIN Artist ar  ON ar.ArtistId= al.ArtistId
GROUP BY t.TrackId
ORDER BY revenue DESC
LIMIT 5
""", conn)
print(top5.to_string(index=False))


Expected (ties at the top are common — unit prices cluster at 0.99×qty):

| track | album | artist | revenue |
|---|---|---|---|
| The Woman King | Battlestar Galactica, Season 3 | Battlestar Galactica | 3.98 |
| The Fix | Heroes, Season 1 | Heroes | 3.98 |
| Walkabout | Lost, Season 1 | Lost | 3.98 |
| Hot Girl | The Office, Season 1 | The Office | 3.98 |
| Gay Witch Hunt | The Office, Season 3 | The Office | 3.98 |

In [ ]:
top5.to_csv("top_tracks.csv", index=False)
print("wrote top_tracks.csv")


---

### 3. Customers with no purchases (LEFT JOIN anti-pattern)

In [ ]:
no_buy = pd.read_sql("""
SELECT c.CustomerId, c.FirstName, c.LastName, c.Email
FROM Customer c
LEFT JOIN Invoice i ON i.CustomerId = c.CustomerId
WHERE i.InvoiceId IS NULL
ORDER BY c.CustomerId
""", conn)
print(len(no_buy), "customers with zero invoices")
# 0 customers with zero invoices  (every Chinook customer has purchased)


> Empty result is a valid answer — document it. The pattern (LEFT JOIN + `IS NULL`) is how you find orphans for any schema.

Equivalent NOT IN:

In [ ]:
no_buy2 = pd.read_sql("""
SELECT * FROM Customer
WHERE CustomerId NOT IN (SELECT DISTINCT CustomerId FROM Invoice)
""", conn)
assert len(no_buy) == len(no_buy2)


**Variant for a non-empty demo** — customers whose total spend is below the median:

In [ ]:
cust_spend = pd.read_sql("""
SELECT c.CustomerId,
       c.FirstName || ' ' || c.LastName AS name,
       COALESCE(ROUND(SUM(i.Total), 2), 0) AS spend
FROM Customer c
LEFT JOIN Invoice i ON i.CustomerId = c.CustomerId
GROUP BY c.CustomerId
ORDER BY spend
""", conn)
print(cust_spend.head(5).to_string(index=False))
print("median spend:", cust_spend["spend"].median())


---

### 4. Monthly revenue trend

In [ ]:
monthly = pd.read_sql("""
SELECT strftime('%Y-%m', InvoiceDate) AS ym,
       ROUND(SUM(Total), 2) AS revenue,
       COUNT(*) AS invoices
FROM Invoice
GROUP BY ym
ORDER BY ym
""", conn)
print(monthly.head(6).to_string(index=False))
print("months:", len(monthly), "total:", monthly["revenue"].sum().round(2))
# months: 60, total ≈ 2328.60


Plot:

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(monthly["ym"], monthly["revenue"], marker="o", ms=3)
ax.set_title("Chinook monthly revenue")
ax.set_ylabel("Revenue ($)")
plt.xticks(rotation=90, fontsize=6)
fig.tight_layout(); fig.savefig("monthly_revenue.png", dpi=120)
print("saved monthly_revenue.png")


Peaks to notice: **2022-01 (52.62)**, **2023-04 (51.62)**, **2023-06 (50.62)**; a dip at **2023-11 (23.76)**.

---

### 5. Bonus: top genres by revenue

In [ ]:
genres = pd.read_sql("""
SELECT g.Name AS genre,
       ROUND(SUM(il.Quantity * il.UnitPrice), 2) AS revenue
FROM InvoiceLine il
JOIN Track t ON t.TrackId = il.TrackId
JOIN Genre g ON g.GenreId = t.GenreId
GROUP BY g.GenreId
ORDER BY revenue DESC
LIMIT 3
""", conn)
print(genres.to_string(index=False))
# Rock    826.65
# Latin   382.14
# Metal   261.36


In [ ]:
conn.close()


---

## Exercises (do these!)

### Exercise 1 — Top 5 tracks by revenue
Run the Section 2 query. List track name and revenue for all 5.
*Expected: five tracks each with revenue 3.98 — The Woman King, The Fix, Walkabout, Hot Girl, Gay Witch Hunt (ties).*

**Follow-up:** What is mean monthly revenue? Check: 38.81.

<details>
<summary>Hint</summary>

Join `InvoiceLine → Track → Album → Artist`, `SUM(Quantity*UnitPrice)`, `ORDER BY revenue DESC LIMIT 5`.
</details>

### Exercise 2 — Customers with no purchases
LEFT JOIN customers to invoices; count rows where invoice is NULL.
*Expected: 0 (all 59 customers have ≥1 invoice). Print 0 and explain the anti-pattern.*

**Follow-up:** What share of revenue is Rock? Check: 0.355.

<details>
<summary>Hint</summary>

`WHERE i.InvoiceId IS NULL` after `LEFT JOIN Invoice`.
</details>

### Exercise 3 — Monthly revenue trend
Group invoices by `strftime('%Y-%m', InvoiceDate)`. How many months? What is total revenue?
*Expected: 60 months · total ≈ 2328.60.*

**Follow-up:** Compare mean vs median customer spend — which way is it skewed? Check: 39.47 above 37.62, right-skewed.

<details>
<summary>Hint</summary>

`SELECT strftime('%Y-%m', InvoiceDate) AS ym, ROUND(SUM(Total),2) … GROUP BY ym ORDER BY ym`.
</details>

---

## Solutions

In [ ]:
# --- Solution 1 ---
import sqlite3, os, pandas as pd
conn = sqlite3.connect("chinook.sqlite")
q1 = """
SELECT t.Name AS track, al.Title AS album, ar.Name AS artist,
       ROUND(SUM(il.Quantity*il.UnitPrice),2) AS revenue
FROM InvoiceLine il
JOIN Track t ON t.TrackId=il.TrackId
JOIN Album al ON al.AlbumId=t.AlbumId
JOIN Artist ar ON ar.ArtistId=al.ArtistId
GROUP BY t.TrackId ORDER BY revenue DESC LIMIT 5
"""
print(pd.read_sql(q1, conn).to_string(index=False))
# all revenues 3.98

# --- Solution 2 ---
q2 = """
SELECT COUNT(*) AS n FROM Customer c
LEFT JOIN Invoice i ON i.CustomerId=c.CustomerId
WHERE i.InvoiceId IS NULL
"""
print(pd.read_sql(q2, conn)["n"][0])  # 0

# --- Solution 3 ---
q3 = """
SELECT strftime('%Y-%m', InvoiceDate) AS ym,
       ROUND(SUM(Total),2) AS revenue
FROM Invoice GROUP BY ym ORDER BY ym
"""
m = pd.read_sql(q3, conn)
print(len(m), round(m["revenue"].sum(), 2))  # 60 2328.6
conn.close()

# --- Follow-up 1 ---
import sqlite3 as _sq3
_c2 = _sq3.connect("chinook.sqlite")
_tot = pd.read_sql("SELECT ROUND(SUM(Total),2) AS t FROM Invoice", _c2).iloc[0, 0]
avg = round(_tot / 60, 2)
print(avg)  # 38.81
assert avg == 38.81
_c2.close()

# --- Follow-up 2 ---
import sqlite3 as _sq3
_c2 = _sq3.connect("chinook.sqlite")
_rk = pd.read_sql("SELECT ROUND(SUM(il.Quantity*il.UnitPrice),2) AS r FROM InvoiceLine il JOIN Track t ON t.TrackId=il.TrackId JOIN Genre g ON g.GenreId=t.GenreId WHERE g.Name='Rock'", _c2).iloc[0, 0]
sh = round(_rk / 2328.6, 4)
print(_rk, sh)  # 826.65 0.355
assert _rk == 826.65 and sh == 0.355
_c2.close()

# --- Follow-up 3 ---
mean_spend = round(2328.6 / 59, 2)
print(mean_spend)  # 39.47 > median 37.62 -> right-skewed
assert mean_spend == 39.47


### What to learn next
- Window functions: `SUM(...) OVER (PARTITION BY …)` for running revenue.
- CTEs (`WITH`) for multi-step pivots.
- EXPLAIN QUERY PLAN on the invoice tables.
- Then SQLAlchemy / DuckDB for larger-than-memory analytics.
- Cheat sheet: connect → explore `sqlite_master` → JOIN path → GROUP BY → export.

*Files in this folder: `chinook.sqlite` · optional outputs `top_tracks.csv`, `monthly_revenue.png`.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
